### Preprocesamiento y Codificación de Variables (Feature Encoding)

Para preparar los datos para los algoritmos de Machine Learning sin comprometer la legibilidad del reporte de negocio, se implementó una estrategia de codificación diferenciada:

#### 1. Carga del Dataset Procesado 📥
* Se importa `credit_risk_dataset_processed.csv` desde `data/processed/`, asegurando que los datos ya cuentan con tratamiento de nulos y depuración de valores atípicos provenientes del EDA.

#### 2. Inspección y Codificación Ordinal / Binaria (`.map`) 🏷️
* **`loan_grade` (Ordinal):** Al representar una escala jerárquica de riesgo crediticio, se aplicó un mapeo numérico ascendente ($A=1$ hasta $G=7$). Esto preserva la relación de orden para que el modelo interprete que un grado superior implica mayor riesgo.
* **`cb_person_default_on_file` (Binaria):** Al ser un indicador booleano de historial crediticio previo, se transformó directamente a formato binario ($N=0$, $Y=1$).

#### 3. Codificación One-Hot Encoding (`pd.get_dummies`) 🔲
* **`person_home_ownership`** y **`loan_intent` (Nominales):** Al carecer de una jerarquía intrínseca, se utilizó One-Hot Encoding para generar variables dummy binarias ($0$ o $1$). Esta técnica evita que el algoritmo asuma relaciones cuantitativas o de prioridad arbitrarias entre categorías independientes.

#### 4. Separación de Entornos (`df` vs. `df_ml`) ⚙️
* Se consolida la versión transformada en `df_ml` exclusivamente con atributos numéricos para el entrenamiento, preservando el DataFrame base `df` con las etiquetas descriptivas originales para su posterior integración y consumo analítico en **Power BI**.

In [10]:
import pandas as pd
import numpy as np
df = pd.read_csv('../data/processed/credit_risk_dataset_cleaned.csv')

df



,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
1,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
2,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
3,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
4,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2
...,...,...,...,...,...,...,...,...,...,...,...,...
32569,57,53000,MORTGAGE,1.0,PERSONAL,C,5800,13.16,0,0.11,N,30
32570,54,120000,MORTGAGE,4.0,PERSONAL,A,17625,7.49,0,0.15,N,19
32571,65,76000,RENT,3.0,HOMEIMPROVEMENT,B,35000,10.99,1,0.46,N,28
32572,56,150000,MORTGAGE,5.0,PERSONAL,B,15000,11.48,0,0.10,N,26


In [11]:
display(df['loan_grade'].unique())
display(df['cb_person_default_on_file'].unique())

<StringArray>
['B', 'C', 'A', 'D', 'E', 'F', 'G']
Length: 7, dtype: str

<StringArray>
['N', 'Y']
Length: 2, dtype: str

In [12]:

df['loan_grade_encoded'] = df['loan_grade'].map({
    'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7})

df['cb_person_default_on_file_encoded'] = df['cb_person_default_on_file'].map({
    'Y' : 1, 'N': 0})


In [13]:
df_ml = pd.get_dummies(data = df, columns = ['person_home_ownership' , 'loan_intent'] , dtype = int)

df_ml = df_ml.drop(columns = ['loan_grade', 'cb_person_default_on_file'])

### Definición de Variables Predictoras (X) y Objetivo (y)

Separamos el dataset procesado en dos componentes fundamentales para el modelado:

* **Variable objetivo (`y`):** `loan_status`, la etiqueta binaria que el banco necesita anticipar (`0` para clientes cumplidores, `1` para morosos).
* **Matriz de características (`X`):** El conjunto de variables socioeconómicas y crediticias del solicitante. Se excluye `loan_status` para evitar **fuga de datos** (*data leakage*).

### Partición de Datos en Entrenamiento y Prueba (Train/Test Split)

Dividimos los datos para simular un escenario de producción real y evaluar el modelo sobre solicitudes que nunca vio durante el aprendizaje:

* **Partición 80/20 (`test_size=0.2`):** El 80% de los datos se destina al aprendizaje y el 20% restante (6.515 registros) a la auditoría final.
* **Estratificación (`stratify=y`):** Garantiza que la proporción original de morosos (~22%) se conserve idéntica tanto en entrenamiento como en prueba, evitando desbalances accidentales.
* **Semilla fija (`random_state=42`):** Asegura la reproducibilidad exacta de los resultados en futuras ejecuciones.

In [14]:
y = df_ml['loan_status']
X = df_ml.drop(columns = ['loan_status'])

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 , stratify = y)

In [16]:
print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_test:", X_test.shape)
print("\nDistribucion de clases en y_train:\n", y_train.value_counts(normalize=True))

Dimensiones de X_train: (26059, 19)
Dimensiones de X_test: (6515, 19)

Distribucion de clases en y_train:
 loan_status
0    0.781803
1    0.218197
Name: proportion, dtype: float64


### 3. Modelo Random Forest y Ajuste de Política de Decisión (Umbral 0.35)

Implementacion de Random Forest y se ajusto el criterio de corte para priorizar la protección del capital:

* **Algoritmo Random Forest:** Modelo no lineal inmune a diferencias de escala que combina cientos de árboles de decisión en paralelo para reducir la varianza.
* **Probabilidad de Impago (`predict_proba`):** En lugar de forzar una etiqueta rígida, calculamos la probabilidad continua de que cada solicitante entre en mora.
* **Ajuste de Umbral al 35% (`umbral = 0.35`):** Modificamos el corte estándar del 50%. En gestión de riesgo financiero, un solicitante con 35% de probabilidad de impago ya representa un riesgo inaceptable; este ajuste eleva el **Recall** al 75% atrapando a la gran mayoría de morosos sin deteriorar la precisión general.

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

modelo_rf = RandomForestClassifier(random_state=42)

modelo_rf.fit(X_train, y_train)

prob_mora = modelo_rf.predict_proba(X_test)[:, 1]

umbral = 0.35

y_pred_rf = (prob_mora >= umbral).astype(int)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.93      0.98      0.95      5094
           1       0.90      0.75      0.81      1421

    accuracy                           0.93      6515
   macro avg       0.91      0.86      0.88      6515
weighted avg       0.92      0.93      0.92      6515



Generación de Scores Crediticios y Exportación de Resultados

Integramos las predicciones del modelo al conjunto evaluado para crear la base operativa de decisión:
* **`probabilidad_mora`:** Score de riesgo continuo generado por el Random Forest.
* **`decision_credito`:** Regla de negocio aplicada con el umbral optimizado al 35% (`Rechazado` para $\ge 0.35$, `Aprobado` para $< 0.35$).
* **Exportación:** Guardado de la base final en formato `.csv` para su consumo en tableros de control o sistemas de originación.

In [19]:
df_final = df_ml.copy()

prob_total = modelo_rf.predict_proba(X)[:, 1]
df_final['probabilidad_mora'] = prob_total.round(4)
df_final['prediccion_mora'] = (prob_total >= umbral).astype(int)

df_final['decision_crediticia'] = df_final['prediccion_mora'].map({
    0: 'Aprobar',
    1: 'Rechazar'
})

df_final.to_csv('../data/processed/credit_risk_final_dataset.csv', index=False)
print("Predicciones guardadas en '../data/processed/credit_risk_final_dataset.csv'")

Predicciones guardadas en '../data/processed/credit_risk_final_dataset.csv'
